In [ ]:
!pip install livelossplot

In [ ]:
# imports that the notebook needs
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import SGD
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from livelossplot import PlotLosses

In [ ]:
# Loading the data and some preprocessing:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
           'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
           'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, names=columns, skipinitialspace=True).sample(5000, random_state=499) # I sampled the data so that it can be used in <1h

# Separate features and target (and drop some problematic variables)
X_raw = df.drop(['income', 'native-country', 'occupation'], axis=1)
y_raw = LabelEncoder().fit_transform(df['income'])

# We explicitly define which columns get scaled vs. which get one-hot encoded
numeric_features = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'marital-status', 'relationship', 'race', 'sex']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features), # Standardize to mean=0, std=1
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features) #one hot encode categorical data
    ])

X_processed = pd.DataFrame(preprocessor.fit_transform(X_raw).astype(np.float32))
y_processed = pd.DataFrame(y_raw.astype(np.float32))

# split off 500 rows for testing
X_train_all, X_test, y_train_all, y_test = train_test_split(
    X_processed, y_processed, test_size=500, random_state=499
)

# split off 500 rows for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all, test_size=500, random_state=499
)

print(X_train.head())

In [ ]:
# defining the model
class FullyConnectedNN(nn.Module):
    def __init__(self, input_dim):
        super(FullyConnectedNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.ReLU = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.ReLU(x)
        x = self.fc2(x)
        x = self.ReLU(x)
        x = self.fc3(x)
        return self.sigmoid(x)

In [ ]:
# convert our data to Torch tensors
X_train_torch = torch.tensor(X_train.values, dtype=torch.float32)
y_train_torch = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_val_torch = torch.tensor(X_val.values, dtype=torch.float32)
y_val_torch = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

# Initialize the model we defined above
model = FullyConnectedNN(input_dim=X_train.shape[1])

# loss and optimizer
criterion = nn.BCELoss()
optimizer = SGD(model.parameters(), lr=) # <------------------------------------------

# initialize loss plot object
plotlosses = PlotLosses()

# training loop
epochs = # <------------------------------------------

for epoch in range(epochs):

    y_train_pred = model(X_train_torch)
    loss = criterion(y_train_pred, y_train_torch)

    with torch.no_grad():
        train_labels = (y_train_pred > 0.5).float()
        train_accuracy = (train_labels == y_train_torch).float().mean()
        y_val_pred = model(X_val_torch)
        val_loss = criterion(y_val_pred, y_val_torch)
        val_labels = (y_val_pred > 0.5).float()
        val_accuracy = (val_labels == y_val_torch).float().mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    plotlosses.update({
        'loss': loss.detach().item(),
        # 'accuracy': train_accuracy,
        'val_loss': val_loss.detach().item(),
        # 'val_accuracy': val_accuracy,
    })
    if (epoch + 1) % 1000 == 0:
        plotlosses.send()

    # if (epoch + 1) % 50 == 0:
    #     with torch.no_grad():
    #         print(f'Epoch {epoch+1}: Train Loss = {loss.item():.5f}, Train Acc = {train_accuracy.item():.2f}, Val Acc = {val_accuracy.item():.2f}')

In [ ]:
# test on the test data
X_test_torch = torch.tensor(X_test.values, dtype=torch.float32)
y_test_torch = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

y_pred = model(X_test_torch)
test_labels = (y_pred > 0.5).float()
test_accuracy = (test_labels == y_test_torch).float().mean()

print(f'Test Accuracy: {test_accuracy.item():.2f}')